# Pruning Some Layer

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images / 255.0
test_images = test_images / 255.0

11490434/11490434 [==============================] - 1s 0us/step


## Load CNN Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Check weight before pruning (dense layer)

In [7]:
model.layers[6].get_weights()

[array([[-0.11423343,  0.07195434,  0.18450372, ...,  0.03865412,
         -0.21373557, -0.04995057],
        [-0.0808438 , -0.03223755,  0.13059856, ..., -0.00118522,
          0.12131715, -0.12105928],
        [ 0.01488143,  0.01744528,  0.11621684, ...,  0.02214462,
          0.05873864, -0.03940482],
        ...,
        [-0.09544836,  0.32731798, -0.08578021, ..., -0.23767424,
          0.09664504, -0.08062645],
        [ 0.24474329, -0.02343009, -0.13883138, ...,  0.1283401 ,
         -0.22653243,  0.01459998],
        [-0.06368535,  0.01293916, -0.04990869, ...,  0.124214  ,
         -0.2552875 ,  0.04953326]], dtype=float32),
 array([ 0.01380989, -0.04487011, -0.03427422,  0.0264946 , -0.00351122,
         0.01776556, -0.05953237, -0.03990521,  0.02347567,  0.06879976,
        -0.01733078,  0.05501936, -0.05372361,  0.01816729,  0.06208762,
         0.0098527 , -0.08965034, -0.04478165, -0.01879789, -0.01960733,
         0.04681579, -0.0102757 , -0.0579581 ,  0.0237167 , -0.054

## Fine-tune pre-trained model with pruning (Some Layer)
* 적용 Layer : 마지막 2개의 Dense Layer에 적용
    * Scheduler : tfmot.sparsity.keras.PolynomialDecay
        * Initial sparsity : 50% (50% zeros in weights)
        * End sparsigy : 80% sparsity.

In [8]:
batch_size = 128
epochs = 2
validation_split = 0.1

num_images = train_images.shape[0] * (1 - validation_split)
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs

pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,
                                                               final_sparsity=0.80,
                                                               begin_step=0,
                                                               end_step=end_step)
}

In [9]:
def apply_pruning_to_dense(layer):
  if isinstance(layer, tf.keras.layers.Dense):
    # Dense layer는 pruning 적용 layer return
    return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params)
  return layer

model_for_pruning = keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_dense,
)

model_for_pruning.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

model_for_pruning.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [10]:
callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep(),
]

hist_pruning = model_for_pruning.fit(train_images, train_labels,
                  batch_size=batch_size, epochs=epochs, validation_split=validation_split,
                  callbacks=callbacks)

Epoch 1/2
422/422 [==============================] - 31s 67ms/step - loss: 0.0113 - accuracy: 0.9963 - val_loss: 0.0389 - val_accuracy: 0.9907
Epoch 2/2
422/422 [==============================] - 28s 66ms/step - loss: 0.0140 - accuracy: 0.9954 - val_loss: 0.0341 - val_accuracy: 0.9915


In [11]:
_, model_for_pruning_accuracy = model_for_pruning.evaluate(
   test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', model_for_pruning_accuracy)

Baseline test accuracy: 0.9900000095367432
Pruned test accuracy: 0.9901000261306763


In [12]:
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

model_for_export.layers[6].get_weights()

[array([[-0.        ,  0.        ,  0.19672833, ..., -0.        ,
         -0.17374958, -0.        ],
        [ 0.        ,  0.        ,  0.        , ..., -0.        ,
          0.        , -0.        ],
        [-0.        ,  0.        , -0.        , ..., -0.        ,
          0.        , -0.        ],
        ...,
        [ 0.        ,  0.3490144 , -0.        , ..., -0.18861821,
          0.        , -0.        ],
        [ 0.26529807,  0.        , -0.        , ...,  0.        ,
          0.        ,  0.        ],
        [-0.        ,  0.        , -0.        , ...,  0.        ,
         -0.24007222,  0.        ]], dtype=float32),
 array([ 0.02474579, -0.02203608, -0.03635075,  0.05579012, -0.02578036,
        -0.02084653, -0.06356259, -0.00465682,  0.04047446,  0.08799312,
        -0.02159246,  0.09305168,  0.00176443,  0.01399403,  0.07630167,
         0.03256657, -0.07129096, -0.01687119, -0.01879789, -0.01961204,
         0.04057411,  0.04348741, -0.03689703,  0.05931766, -0.046

In [13]:
total, non_zero = 0, 0

for l in model_for_export.layers:
    weights = l.get_weights()
    for i, w in enumerate(weights):
        if type(w) == np.ndarray:
            rate = (w.size - np.count_nonzero(w))/w.size
            print("layer [{}] / weight [{}] : rate = {}".format(l.name, i, rate))
            total += w.size
            non_zero += np.count_nonzero(w)

print( "=" * 40)
print( "Total parameter : {}".format(total))
print( "Non-zero parameter : {}".format(non_zero))
print( "Rate of pruned parmeter : {}".format((total-non_zero)/ total))

layer [conv2d] / weight [0] : rate = 0.0
layer [conv2d] / weight [1] : rate = 0.0
layer [conv2d_1] / weight [0] : rate = 0.0
layer [conv2d_1] / weight [1] : rate = 0.0
layer [dense] / weight [0] : rate = 0.7999609375
layer [dense] / weight [1] : rate = 0.0
layer [dense_1] / weight [0] : rate = 0.8
layer [dense_1] / weight [1] : rate = 0.0
Total parameter : 57562
Non-zero parameter : 15580
Rate of pruned parmeter : 0.729335325388277


In [14]:
model_for_export.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0